# U-Net, 5 каналов

```
TermoDataset (T,H,W)
  → SelectFrames → augs → Stack([MaxMin, Std, PCA1, TSRDeriv×2]) → PercentileNorm
  → (5, 256, 256)
  → UNetModel → mask
```

| # | канал | смысл |
|--:|---|---|
| 1 | MaxMin | max(t) − min(t) |
| 2 | Std | σ(t) |
| 3 | PCA1 | 1-я временная мода |
| 4 | TSRDeriv(1) | max \|p′\| по остыванию |
| 5 | TSRDeriv(2) | max \|p″\| по остыванию |

Сеть — классический U-Net (как `models/U-Net`), loss — `BCEDiceLoss`.
После каждой эпохи: IoU в консоль; графики loss / Dice; превью PCA1 · GT · pred.


In [ ]:
# === configuration ===
INCLUDE = None
NUM_FRAMES = 128
EPOCHS = 50
BATCH_SIZE = 4
TEST_EVERY = 4
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 10.0
NUM_WORKERS = 0
PREVIEW_EVERY = 1


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pytorch_lightning as pl
import torch
from IPython.display import clear_output, display
from torch.utils.data import DataLoader, Subset

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "datasets.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "datasets"))

from datasets import TermoDataset
from transforms import (
    Compose, Stack, SelectFrames, RandomChoice,
    HorizontalFlip, VerticalFlip, Transpose, RandomRotate90,
    MaxMin, Std, PCA1, TSRDeriv, PercentileNorm,
)
from models.segmentation import UNetModel, BCEDiceLoss, SegmentationLightningModule

extractors = [MaxMin(), Std(), PCA1(), TSRDeriv(order=1), TSRDeriv(order=2)]
IN_CHANNELS = Stack(extractors).out_channels()
assert IN_CHANNELS == 5, IN_CHANNELS

augs = RandomChoice(
    [HorizontalFlip(), VerticalFlip(), Transpose(), RandomRotate90()],
    k=2,
)
train_tf = Compose([SelectFrames(NUM_FRAMES), augs, Stack(extractors), PercentileNorm()])
val_tf = Compose([SelectFrames(NUM_FRAMES), Stack(extractors), PercentileNorm()])

print(f"in_channels={IN_CHANNELS}  {[type(e).__name__ for e in extractors]}")


## 1. Данные и сплит по видео


In [ ]:
DATA_ROOT = str(ROOT / "datasets" / "datasets_list")
train_full = TermoDataset(DATA_ROOT, include=INCLUDE, transform=train_tf)
val_full = TermoDataset(DATA_ROOT, include=INCLUDE, transform=val_tf)
assert [p for p, *_ in train_full.items] == [p for p, *_ in val_full.items]

n = len(train_full)
val_idx = list(range(0, n, TEST_EVERY))
train_idx = [i for i in range(n) if i not in set(val_idx)]
train_ds = Subset(train_full, train_idx)
val_ds = Subset(val_full, val_idx)

print(f"train {len(train_ds)}  val {len(val_ds)}")
x, y = train_ds[0]
print(f"item: {tuple(x.shape)} mask {tuple(y.shape)}")
assert x.shape[0] == 5


## 2. Train

После каждой эпохи: **IoU в консоль**; графики — **loss / Dice** (train+val); превью — **PCA1 · GT · pred**.
Чекпоинт — лучший `val_iou` в `runs/seg_5ch/`.


In [ ]:
def _metric(m, key):
    v = m.get(key)
    return float(v.detach().cpu()) if v is not None else float("nan")


class LivePlot(pl.Callback):
    """IoU → console only; plots loss / Dice; preview PCA1 · GT · pred."""

    def __init__(self, preview_loader, every: int = 1):
        super().__init__()
        self.preview_loader = preview_loader
        self.every = every
        self.history = []
        self._train = {}

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        m = trainer.callback_metrics
        self._train = {
            "train_loss": _metric(m, "train_loss"),
            "train_iou": _metric(m, "train_iou"),
            "train_dice": _metric(m, "train_dice"),
        }

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        m = trainer.callback_metrics
        row = {
            "epoch": int(trainer.current_epoch) + 1,
            **self._train,
            "val_loss": _metric(m, "val_loss"),
            "val_iou": _metric(m, "val_iou"),
            "val_dice": _metric(m, "val_dice"),
        }
        self.history.append(row)
        print(
            f"epoch {row['epoch']:3d}  "
            f"train loss {row['train_loss']:.4f}  val loss {row['val_loss']:.4f}  "
            f"train IoU {row['train_iou']:.4f}  val IoU {row['val_iou']:.4f}"
        )

        if row["epoch"] % self.every and row["epoch"] != trainer.max_epochs:
            return

        clear_output(wait=True)
        best = max(self.history, key=lambda r: r["val_iou"])
        print(
            f"epoch {row['epoch']}/{trainer.max_epochs}    "
            f"best val IoU {best['val_iou']:.4f} @ {best['epoch']}    "
            f"train IoU {row['train_iou']:.4f}  val IoU {row['val_iou']:.4f}"
        )

        epochs = [r["epoch"] for r in self.history]
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
        axes[0].plot(epochs, [r["train_loss"] for r in self.history], label="train")
        axes[0].plot(epochs, [r["val_loss"] for r in self.history], label="val")
        axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

        axes[1].plot(epochs, [r["train_dice"] for r in self.history], label="train")
        axes[1].plot(epochs, [r["val_dice"] for r in self.history], label="val")
        axes[1].set_title("Dice"); axes[1].legend(); axes[1].grid(alpha=0.3)
        for ax in axes:
            ax.set_xlabel("epoch")
        fig.tight_layout()
        display(fig)
        plt.close(fig)

        pl_module.eval()
        with torch.no_grad():
            xb, yb = next(iter(self.preview_loader))
            xb = xb.to(pl_module.device)
            prob = torch.sigmoid(pl_module(xb)).cpu()
        x0, y0, p0 = xb[0].cpu(), (yb[0] > 0).float(), prob[0, 0]
        fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
        axs[0].imshow(x0[2].numpy(), cmap="inferno"); axs[0].set_title("PCA1")
        axs[1].imshow(y0.numpy() if y0.ndim == 2 else y0[0].numpy(), cmap="gray"); axs[1].set_title("GT")
        axs[2].imshow(p0.numpy(), cmap="gray", vmin=0, vmax=1); axs[2].set_title("pred")
        for ax in axs:
            ax.axis("off")
        fig.tight_layout()
        display(fig)
        plt.close(fig)
        pl_module.train()


In [ ]:
train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
preview_ld = DataLoader(val_ds, batch_size=1, shuffle=False)

model = SegmentationLightningModule(
    UNetModel(in_channels=IN_CHANNELS, num_classes=1),
    BCEDiceLoss(pos_weight=POS_WEIGHT),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

out = Path("runs/seg_5ch")
out.mkdir(parents=True, exist_ok=True)
ckpt_cb = pl.callbacks.ModelCheckpoint(
    dirpath=out, filename="best", monitor="val_iou", mode="max", save_last=True,
)
live = LivePlot(preview_ld, every=PREVIEW_EVERY)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    callbacks=[ckpt_cb, live],
    enable_checkpointing=True,
    log_every_n_steps=1,
)
trainer.fit(model, train_ld, val_ld)
print("best val IoU:", float(ckpt_cb.best_model_score or 0))
